# Necessary imports

In [2]:
import os
import re
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Templates of subs

# By using lorentzian function we can get a clean spectra looking function in which we add randomness and noise in order to create entry for our dataset

In [7]:
def slugify(name: str) -> str:
    """Safe folder name from compound."""
    name = name.strip().lower()
    name = re.sub(r"\s+", "_", name)
    name = re.sub(r"[^a-z0-9_\-\.]", "", name)
    return name or "unknown"

def lorentzian(x, x0, gamma):
    # gamma = HWHM in ppm
    return (gamma**2) / ((x - x0)**2 + gamma**2)

def generate_spectrum(
    peaks_ppm: np.ndarray,
    peaks_int: np.ndarray,
    ppm_min: float = -0.5,
    ppm_max: float = 10.5,
    n_points: int = 8192,
    rng: np.random.Generator | None = None,
):
    rng = np.random.default_rng() if rng is None else rng

    x = np.linspace(ppm_min, ppm_max, n_points).astype(np.float32)
    y = np.zeros_like(x)

    # realistická referencing chyba
    global_shift = rng.uniform(-0.03, 0.03)

    # Global intensity scale
    global_scale = rng.uniform(0.6, 1.4)

    for p, a in zip(peaks_ppm, peaks_int):
        # small per-peak shifts and intensity jitter
        pj = (p + global_shift + rng.normal(0, 0.002)).astype(np.float32)
        aj = (a * global_scale * rng.uniform(0.85, 1.15)).astype(np.float32)

        # linewidth (HWHM) in ppm
        gamma = rng.uniform(0.0015, 0.006)

        y += aj * lorentzian(x, pj, gamma)

        # # Optional: fake multiplet effect (very small split sometimes)
        # if rng.random() < 0.15:
        #     split = rng.uniform(0.002, 0.01)  # ppm
        #     y += (aj * 0.35) * lorentzian(x, pj - split, gamma * 1.1)
        #     y += (aj * 0.35) * lorentzian(x, pj + split, gamma * 1.1)

    # Contaminants (optional)
    if rng.random() < 0.7:
        # water
        y += rng.uniform(0.05, 0.2) * lorentzian(x, 4.70 + rng.normal(0, 0.02), rng.uniform(0.002, 0.02))
    if rng.random() < 0.7:
        # CDCl3
        y += rng.uniform(0.02, 0.15) * lorentzian(x, 7.26 + rng.normal(0, 0.01), rng.uniform(0.001, 0.01))

    # Baseline drift (smooth random curve)
    baseline_scale = rng.uniform(0.0, 0.05)
    
    noise = rng.normal(0, 1, n_points)
    kernel = np.ones(200) / 200  # smooth it
    smooth_noise = np.convolve(noise, kernel, mode="same")
    
    baseline = baseline_scale * smooth_noise
    
    y += baseline

    # Noise
    noise_sigma = rng.uniform(0.001, 0.01)
    y += rng.normal(0, noise_sigma, size=n_points).astype(np.float32)

    # Robust normalize
    y = y - np.median(y)
    p99 = np.percentile(np.abs(y), 99)
    y = y / (p99 + 1e-6)

    return x, y.astype(np.float32)

def save_png(x, y, path_png, dpi=150):
    plt.figure(figsize=(6, 2))
    plt.plot(x, y, linewidth=1)
    plt.gca().invert_xaxis()
    plt.axis("off")
    plt.savefig(path_png, dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close()

# --------- GENERATOR ---------

def build_dataset(
    csv_path: str,
    out_dir: str = "dataset_out",
    samples_per_compound: int = 500,
    save_vectors: bool = True,
    save_images: bool = True,
    seed: int = 0,
):
    rng = np.random.default_rng(seed)
    df = pd.read_csv(CSV_PATH)

    required = {"compound", "ppm", "intensity"}
    if not required.issubset(df.columns):
        raise ValueError(f"CSV must contain columns: {required} (found {set(df.columns)})")

    # CISTENIE DATASETU
    df = df.dropna(subset=["compound", "ppm", "intensity"]).copy()
    df["compound"] = df["compound"].astype(str)

    os.makedirs(out_dir, exist_ok=True)

    # LABEL MAPA LOGIKA
    compounds = sorted(df["compound"].unique().tolist())
    label_map = {c: i for i, c in enumerate(compounds)}
    with open(os.path.join(out_dir, "label_map.json"), "w", encoding="utf-8") as f:
        json.dump(label_map, f, ensure_ascii=False, indent=2)

    # Output roots
    if save_images:
        img_root = os.path.join(out_dir, "images")
        os.makedirs(img_root, exist_ok=True)
    if save_vectors:
        vec_root = os.path.join(out_dir, "vectors")
        os.makedirs(vec_root, exist_ok=True)

    rows = []  # index

    for compound, g in df.groupby("compound", sort=False):
        peaks_ppm = g["ppm"].to_numpy(np.float32)
        peaks_int = g["intensity"].to_numpy(np.float32)

        comp_dirname = slugify(compound)

        if save_images:
            comp_img_dir = os.path.join(img_root, comp_dirname)
            os.makedirs(comp_img_dir, exist_ok=True)
        if save_vectors:
            comp_vec_dir = os.path.join(vec_root, comp_dirname)
            os.makedirs(comp_vec_dir, exist_ok=True)

        for i in range(samples_per_compound):
            x, y = generate_spectrum(peaks_ppm, peaks_int, rng=rng)
            sid = f"{comp_dirname}_{i:05d}"

            png_path = ""
            npy_path = ""

            if save_images:
                png_path = os.path.join(comp_img_dir, f"{sid}.png")
                save_png(x, y, png_path)

            if save_vectors:
                npy_path = os.path.join(comp_vec_dir, f"{sid}.npy")
                np.save(npy_path, y)

            rows.append({
                "sample_id": sid,
                "compound": compound,
                "label": label_map[compound],
                "png_path": png_path,
                "npy_path": npy_path,
            })

        print(f"Done: {compound} -> {samples_per_compound} samples")

    index_df = pd.DataFrame(rows)
    index_df.to_csv(os.path.join(out_dir, "index.csv"), index=False)
    print(f"\nSaved dataset to: {out_dir}")
    print(f"Index: {os.path.join(out_dir, 'index.csv')}")
    print(f"Labels: {os.path.join(out_dir, 'label_map.json')}")

# Example usage:
# build_dataset("peaks.csv", out_dir="nmr_dataset", samples_per_compound=500, save_vectors=True, save_images=True)

if __name__ == "__main__":
    # Change these:
    CSV_PATH = "../spectra_dataset.csv"
    OUT_DIR = "../nmr_dataset"
    build_dataset(CSV_PATH, out_dir=OUT_DIR, samples_per_compound=10, save_vectors=True, save_images=True, seed=42)

Done: ethanol -> 10 samples
Done: acetone -> 10 samples
Done: toluene -> 10 samples

Saved dataset to: ../nmr_dataset
Index: ../nmr_dataset/index.csv
Labels: ../nmr_dataset/label_map.json
